In [96]:
import sys
sys.path.append('..')
from osp import *

In [97]:
df_meta = get_corpus_metadata()
# df_meta.iloc[0]

In [98]:
def get_meta(slice_id):
    text_id = slice_id.split('__')[0]
    return {
        'discipline': df_meta.loc[text_id].discipline,
        'period': df_meta.loc[text_id].period,
    }

In [99]:
df = STASH_SLICE_FEATS.df
df0 = df[[c for c in df.columns if c not in BAD_SLICE_FEATS and not c.startswith('phrase_') and not c.startswith('ttr_')]]
df0

KeyboardInterrupt: 

In [100]:
dfz = df0.copy().fillna(0)
for c in dfz.columns:
    dfz[c] = (dfz[c] - dfz[c].mean()) / dfz[c].std()

In [101]:
dfm = pd.DataFrame(list(df0.index.map(get_meta)), index=df0.index)
dfm

,discipline,period
_key,,
phil/10.2307/2380200__04,Philosophy,1950-1975
other/10.2307/2514892__02,Other,1975-2000
other/10.2307/14710__03,Other,1925-1950
phil/10.2307/20111773__02,Philosophy,1950-1975
phil/10.2307/2379466__05,Philosophy,1950-1975
...,...,...
lit/468421__06,Literature,1975-2000
other/10.2307/29782027__24,Other,1900-1925
lit/459531__04,Literature,1950-1975


In [102]:

def shorten_eg(eg, max_len=40):
    eg_pre,w,eg_post = eg.split('*',2)
    radius = (max_len - len(w)) // 2
    eg_pre = eg_pre[-radius:]
    eg_post = eg_post[:max_len-len(eg_pre)-len(w)]
    return f'...{eg_pre}XXXEMPHXXX{{{w.lower()}}}{eg_post}...'


with open('../data/feat2egs.json','r') as f:
    feat2egs = json.load(f)
feat2eg = {feat:shorten_eg(egs[0], 25) for feat,egs in feat2egs.items()}

In [103]:
df0m = df0.join(dfm).query('discipline=="Philosophy"')
dfzm = dfz.join(dfm).query('discipline=="Philosophy"')

In [108]:
df0mT = df0m.groupby('period').mean(numeric_only=True).T
df0mT['1900-1950'] = df0mT[['1900-1925','1925-1950']].mean(axis=1)
df0mT['1950-2000'] = df0mT[['1950-1975','2000-2025']].mean(axis=1)
df0mT['avg'] = df0mT.mean(axis=1)
# df0mT

In [109]:
dfzmT = dfzm.groupby('period').mean(numeric_only=True).T
dfzmT['1900-1950'] = dfzmT[['1900-1925','1925-1950']].mean(axis=1)
dfzmT['1950-2000'] = dfzmT[['1950-1975','2000-2025']].mean(axis=1)
dfzmT['avg'] = dfzmT.mean(axis=1)
dfzmT = dfzmT.sort_values('avg',ascending=False)
dfzmT

period,1900-1925,1925-1950,1950-1975,1975-2000,2000-2025,1900-1950,1950-2000,avg
deprel_cop,0.544264,0.637659,0.770922,0.646852,0.658707,0.590961,0.714814,0.652025
deprel_mark,0.253273,0.252807,0.689594,0.776610,0.860939,0.253040,0.775267,0.551647
pos_MD,0.344272,0.385953,0.643169,0.706137,0.562123,0.365113,0.602646,0.515630
pos_VBZ,0.375948,0.449899,0.561623,0.563624,0.615563,0.412924,0.588593,0.509739
sent_DC,0.217462,0.181653,0.600378,0.674882,0.784578,0.199558,0.692478,0.478713
...,...,...,...,...,...,...,...,...
pos_NNPS,-0.313947,-0.379314,-0.394774,-0.408103,-0.411664,-0.346630,-0.403219,-0.379664
deprel_compound,-0.661316,-0.649352,-0.502670,-0.257641,0.050510,-0.655334,-0.226080,-0.414555
sent_ICw,-0.268377,-0.186758,-0.520091,-0.555896,-0.606514,-0.227568,-0.563303,-0.418358
pos_VBD,-0.391254,-0.392476,-0.421372,-0.524849,-0.584075,-0.391865,-0.502724,-0.458374


In [113]:
odfl = []
for feat,row in dfzmT.head(25).iterrows():
    odx = {
        'Feature': FEAT2DESC[feat],
        'Example': feat2eg.get(feat, ''),
    }
    for k,v in row.items():
        # if k == 'avg':
        #     continue
        if k not in {'1900-1950', '1950-2000', '2000-2025'}:
            continue
        v0 = df0mT.loc[feat,k]
        v2 = f'{v0:.0f} ({"" if v>0 else ""}{v:.2f})'
        # k = k[:5]+k[-2:]
        odx[k] = v2
    odfl.append(odx)
odf = pd.DataFrame(odfl)#.drop()columns=['1900-25'])
odf = odf[['Feature', 'Example', '1900-1950', '1950-2000', '2000-2025']]
odf

,Feature,Example,1900-1950,1950-2000,2000-2025
0,Copula,...this essay XXXEMPHXXX{is} to explore ...,27 (0.59),28 (0.71),27 (0.66)
1,Marker,...ecessary XXXEMPHXXX{because} it lies ...,40 (0.25),48 (0.78),49 (0.86)
2,Modal,... of ours XXXEMPHXXX{should} be exchan...,15 (0.37),17 (0.60),17 (0.56)
3,"Verb, 3rd person sing. pres.",...onclusion XXXEMPHXXX{does} not follow...,40 (0.41),43 (0.59),43 (0.62)
4,# Dependent clauses,,72 (0.20),81 (0.69),83 (0.78)
5,"Verb, base form",...vation to XXXEMPHXXX{imply} that this...,34 (0.22),39 (0.64),39 (0.68)
6,Expletive,"...However, XXXEMPHXXX{it} is possible t...",6 (0.39),6 (0.53),6 (0.52)
7,Wh-determiner,...cial kind XXXEMPHXXX{which} are known...,11 (0.60),10 (0.35),10 (0.34)
8,# Clause transitions,,119 (0.14),129 (0.66),131 (0.79)
9,Adverb,... has not XXXEMPHXXX{perhaps} been ser...,48 (0.35),50 (0.49),50 (0.52)


In [114]:
# df_to_latex_table?

In [115]:
caption = """
The top 25 most distinctively frequent syntactic features of philosophy articles (i.e. the top 25 features with the highest $z$-scores).
Numbers outside parentheses reflect the raw average frequency per 1,000 words.
Numbers inside parentheses reflect the $z$-score (i.e. the number of standard deviations above or below the feature's average frequency in the corpus).
""".replace("\n", " ").strip()

out=df_to_latex_table(odf, caption=caption, label="table:distinctive_features", size="\\footnotesize")
out = out.replace("XXXEMPHXXX\{","\\textbf{")
out = out.replace("\\}","}")
print(out)

\begin{table}[H]
  \centering
  \footnotesize
  \begin{tabular}{lllll}
  \toprule
  Feature & Example & 1900-1950 & 1950-2000 & 2000-2025 \\
  \midrule
  Copula & ...this essay \textbf{is} to explore ... & 27 (0.59) & 28 (0.71) & 27 (0.66) \\
  Marker & ...ecessary \textbf{because} it lies ... & 40 (0.25) & 48 (0.78) & 49 (0.86) \\
  Modal & ... of ours \textbf{should} be exchan... & 15 (0.37) & 17 (0.60) & 17 (0.56) \\
  Verb, 3rd person sing. pres. & ...onclusion \textbf{does} not follow... & 40 (0.41) & 43 (0.59) & 43 (0.62) \\
  \# Dependent clauses &  & 72 (0.20) & 81 (0.69) & 83 (0.78) \\
  Verb, base form & ...vation to \textbf{imply} that this... & 34 (0.22) & 39 (0.64) & 39 (0.68) \\
  Expletive & ...However, \textbf{it} is possible t... & 6 (0.39) & 6 (0.53) & 6 (0.52) \\
  Wh-determiner & ...cial kind \textbf{which} are known... & 11 (0.60) & 10 (0.35) & 10 (0.34) \\
  \# Clause transitions &  & 119 (0.14) & 129 (0.66) & 131 (0.79) \\
  Adverb & ... has not \textbf{perhaps} 

In [116]:
with open('../../../Dropbox/Prof/Articles/OSP/tables/table.topfeats3.tex', 'w') as f:
    f.write(out)
